In [ ]:
using MuJoCo

In [ ]:
"""
Falling simulation function
"""


def simulate_cheetah_falling(
    model: mj.MjModel,
    zoffset: float,
    zidx: int,
    dt: float,
    N: int,
    eps_fd: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Preallocate dynamics jacobians
    A = np.zeros((2 * model.nv, 2 * model.nv))
    B = np.zeros((2 * model.nv, model.nu))

    # Preallocate jacobian-norm trajectory
    Anorm_traj = np.zeros(N)
    Bnorm_traj = np.zeros(N)

    # Preallocate state trajectory
    qtraj = np.zeros((N, model.nq))

    # Set model time step
    model.opt.timestep = dt

    # Init state zoffset above ground
    data = mj.MjData(model)
    mj.mj_resetData(model, data)
    data.qpos[zidx] += zoffset

    # Simulate
    mj.mj_forward(model, data)
    for k in range(N):
        # Get dynamics jacobians
        mj.mjd_transitionFD(model, data, eps_fd, True, A, B, None, None)

        # Update jacobian norms
        Anorm_traj[k] = np.linalg.norm(A)
        Bnorm_traj[k] = np.linalg.norm(B)

        # Update state
        qtraj[k, :] = data.qpos
        mj.mj_step(model, data)

    return qtraj, Anorm_traj, Bnorm_traj

In [ ]:
"""
Robot models and simulation parameters
"""

model_stiff = mj.MjModel.from_xml_path("assets/half_cheetah_stiff.xml")
model_smooth = mj.MjModel.from_xml_path("assets/half_cheetah_smooth.xml")

Z_OFFSET = 0.1
Z_IDX = model_stiff.joint("rootz").qposadr

DT = 1e-2
N = int(1e2)

EPS_FD = 1e-6

In [ ]:
"""
Simulate stiff and smooth models
"""

qtraj_stiff, Anorms_stiff, Bnorms_stiff = simulate_cheetah_falling(
    model=model_stiff,
    zoffset=Z_OFFSET,
    zidx=Z_IDX,
    dt=DT,
    N=N,
    eps_fd=EPS_FD,
)

qtraj_smooth, Anorms_smooth, Bnorms_smooth = simulate_cheetah_falling(
    model=model_smooth,
    zoffset=Z_OFFSET,
    zidx=Z_IDX,
    dt=DT,
    N=N,
    eps_fd=EPS_FD,
)

In [ ]:
"""
Plot dynamics jacobian norms
"""

# Time step indices
ts = np.arange(0.0, N * DT, DT)

# Plot norms as time-series curves
curve_Astiff = go.Scatter(
    x=ts,
    y=Anorms_stiff,
    name="stiff A-norm",
    mode="lines",
    line=dict(color="darkblue", width=2),
)
curve_Bstiff = go.Scatter(
    x=ts,
    y=Bnorms_stiff,
    name="stiff B-norm",
    mode="lines",
    line=dict(color="darkred", width=2),
)
curve_Asmooth = go.Scatter(
    x=ts,
    y=Anorms_smooth,
    name="smooth A-norm",
    mode="lines",
    line=dict(color="royalblue", width=2),
)
curve_Bsmooth = go.Scatter(
    x=ts,
    y=Bnorms_smooth,
    name="smooth B-norm",
    mode="lines",
    line=dict(color="salmon", width=2),
)

# Assemble figure
fig = go.Figure(data=[curve_Astiff, curve_Bstiff, curve_Asmooth, curve_Bsmooth])
fig.update_layout(
    title="Dynamics Jacobian norm trajectories",
    xaxis_title="time (s)",
    yaxis_title="Frobenius norm",
    autosize=False,
    width=16 * 50,
    height=9 * 50,
)
fig.update_yaxes(type="log")
fig.show()

In [ ]:
"""
Plot height trajectory
"""

# Index height states
ztraj_stiff = qtraj_stiff[:, Z_IDX].flatten()
ztraj_smooth = qtraj_smooth[:, Z_IDX].flatten()

# Plot norms as time-series curves
curve_zstiff = go.Scatter(
    x=ts,
    y=ztraj_stiff,
    name="stiff",
    mode="lines",
    line=dict(color="darkblue", width=2),
)
curve_zsmooth = go.Scatter(
    x=ts,
    y=ztraj_smooth,
    name="smooth",
    mode="lines",
    line=dict(color="royalblue", width=2),
)

# Assemble figure
fig = go.Figure(data=[curve_zstiff, curve_zsmooth])
fig.update_layout(
    title="Height trajectories",
    xaxis_title="time (s)",
    yaxis_title="height (m)",
    autosize=False,
    width=16 * 50,
    height=9 * 50,
)
fig.show()